# Pipeline 3 - YOLOv8 + Molmo2-4B VLM

Fine-tuned YOLOv8-seg crops the screen; the Molmo2-4B vision-language model reads HR and SpO2 from the crop in a single inference pass.

## Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip install -q ultralytics transformers==4.57.1 accelerate einops

In [ ]:
import os, re, json
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

DATASET_DIR = "/content/drive/MyDrive/CUBIC/Colab Notebooks/data"
DATA_YAML = os.path.join(DATASET_DIR, "data.yaml")
EVAL_SPLIT = "test"                       # train | valid | test

GT_VITALS_CSV = os.path.join(DATASET_DIR, "ground_truth_vitals.csv")
METRICS_CSV = os.path.join(DATASET_DIR, "pipeline_metrics_summary.csv")

CLASS_NAMES = ["HR", "PM", "SPO2"]        # id order of the Roboflow data.yaml export
HR_CLASS_ID, PM_CLASS_ID, SPO2_CLASS_ID = 0, 1, 2
HR_RANGE, SPO2_RANGE = (40, 200), (70, 100)
COVERAGE_BINS = (0, 5, 10, 20, 35, 50, 75, 100)
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp")

import torch
from PIL import Image
from ultralytics import YOLO
from transformers import AutoModelForImageTextToText, AutoProcessor

MOLMO_MODEL = "allenai/Molmo2-4B"
print("CUDA:", torch.cuda.is_available())

In [ ]:
# Evaluation utilities. This block is identical in all three pipeline notebooks.

def list_images(directory):
    return sorted(f for f in os.listdir(directory) if f.lower().endswith(IMG_EXTS))


def _iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def _dice(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    total = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1])
    return 2 * inter / total if total > 0 else 0.0


def _corner_mae(a, b):
    return float(np.mean([abs(a[i] - b[i]) for i in range(4)]))


def load_screen_box(label_path, img_w, img_h, class_id=PM_CLASS_ID):
    """Largest PM-class box from a YOLO label file (box or polygon format), or None."""
    if not os.path.exists(label_path):
        return None
    boxes = []
    for line in open(label_path):
        parts = line.split()
        if len(parts) < 5 or int(float(parts[0])) != class_id:
            continue
        c = [float(v) for v in parts[1:]]
        if len(c) == 4:
            cx, cy, bw, bh = c
            boxes.append([(cx - bw / 2) * img_w, (cy - bh / 2) * img_h,
                          (cx + bw / 2) * img_w, (cy + bh / 2) * img_h])
        elif len(c) >= 6 and len(c) % 2 == 0:
            xs, ys = c[0::2], c[1::2]
            boxes.append([min(xs) * img_w, min(ys) * img_h, max(xs) * img_w, max(ys) * img_h])
    return max(boxes, key=lambda b: (b[2] - b[0]) * (b[3] - b[1])) if boxes else None


def evaluate_stage1(readings, images_dir, labels_dir):
    """Screen localization: IoU, Dice and mean corner error vs. the PM labels.
    A missing detection counts as IoU = Dice = 0 and is excluded from the MAE."""
    ious, dices, maes = [], [], []
    no_label = no_detection = 0
    for r in readings:
        img = cv2.imread(os.path.join(images_dir, r["image_file"]))
        if img is None:
            continue
        h, w = img.shape[:2]
        gt = load_screen_box(os.path.join(labels_dir, os.path.splitext(r["image_file"])[0] + ".txt"), w, h)
        if gt is None:
            no_label += 1
            continue
        if r["bbox"] is None:
            no_detection += 1
            ious.append(0.0)
            dices.append(0.0)
            continue
        ious.append(_iou(r["bbox"], gt))
        dices.append(_dice(r["bbox"], gt))
        maes.append(_corner_mae(r["bbox"], gt))
    print(f"[Stage 1] scored {len(ious)} images "
          f"({no_label} without a PM label, {no_detection} without a detection)")
    return {"IoU": float(np.mean(ious)) if ious else float("nan"),
            "Dice": float(np.mean(dices)) if dices else float("nan"),
            "Screen_MAE_px": float(np.mean(maes)) if maes else float("nan")}


def _pm_key(filename):
    m = re.match(r"(PM_\d+)", filename)
    return m.group(1) if m else os.path.splitext(filename)[0]


def load_gt_vitals(gt_csv=GT_VITALS_CSV, legible_only=True):
    gt = pd.read_csv(gt_csv)
    gt["_key"] = gt["image" if "image" in gt.columns else "image_file"].map(_pm_key)
    if legible_only and "legible" in gt.columns:
        gt = gt[gt["legible"] == 1]
    return gt.set_index("_key")


def evaluate_stage2(readings, gt_csv=GT_VITALS_CSV, hr_tol=0, spo2_tol=0):
    """Vital extraction: MAE and exact-match accuracy over the readings that
    produced a value and have legible ground truth."""
    gt = load_gt_vitals(gt_csv)
    hr_err, spo2_err, hr_hit, spo2_hit = [], [], 0, 0
    for r in readings:
        key = _pm_key(r["image_file"])
        if key not in gt.index:
            continue
        row = gt.loc[key]
        if r["hr"] is not None and not pd.isna(row["hr_true"]):
            e = abs(r["hr"] - row["hr_true"])
            hr_err.append(e)
            hr_hit += e <= hr_tol
        if r["spo2"] is not None and not pd.isna(row["spo2_true"]):
            e = abs(r["spo2"] - row["spo2_true"])
            spo2_err.append(e)
            spo2_hit += e <= spo2_tol
    print(f"[Stage 2] scored {len(hr_err)} HR and {len(spo2_err)} SpO2 readings")
    return {"HR_MAE_bpm": float(np.mean(hr_err)) if hr_err else float("nan"),
            "HR_Acc_pct": 100 * hr_hit / len(hr_err) if hr_err else float("nan"),
            "SpO2_MAE_pct": float(np.mean(spo2_err)) if spo2_err else float("nan"),
            "SpO2_Acc_pct": 100 * spo2_hit / len(spo2_err) if spo2_err else float("nan")}


def write_metrics_row(pipeline, stage1, stage2, csv=METRICS_CSV):
    row = {"Pipeline": pipeline, **stage1, **stage2}
    df = pd.read_csv(csv) if os.path.exists(csv) else pd.DataFrame()
    if len(df):
        df = df[df["Pipeline"] != pipeline]
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(csv, index=False)
    print(df.to_string(index=False))
    return df


def coverage_curve(readings, images_dir, prefix, gt_csv=GT_VITALS_CSV, bins=COVERAGE_BINS):
    """Bin readings by screen coverage and report exact-match accuracy per bin.
    Writes <prefix>_coverage_bins.csv next to the dataset for the comparison notebook."""
    gt = load_gt_vitals(gt_csv)
    points = []
    for r in readings:
        img = cv2.imread(os.path.join(images_dir, r["image_file"]))
        if img is None or r["bbox"] is None:
            continue
        h, w = img.shape[:2]
        x1, y1, x2, y2 = r["bbox"]
        cov = 100 * (x2 - x1) * (y2 - y1) / (h * w)
        row = gt.loc[_pm_key(r["image_file"])] if _pm_key(r["image_file"]) in gt.index else None
        hr_ok = spo2_ok = np.nan
        if row is not None and not pd.isna(row["hr_true"]):
            hr_ok = float(r["hr"] is not None and r["hr"] == row["hr_true"])
        if row is not None and not pd.isna(row["spo2_true"]):
            spo2_ok = float(r["spo2"] is not None and r["spo2"] == row["spo2_true"])
        points.append((cov, hr_ok, spo2_ok))

    rows = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        sel = [(hr, sp) for cov, hr, sp in points if lo <= cov < hi]
        hr_vals = [h for h, s in sel if not np.isnan(h)]
        sp_vals = [s for h, s in sel if not np.isnan(s)]
        both = [h * s for h, s in sel if not np.isnan(h) and not np.isnan(s)]
        rows.append({"bin_mid_pct": (lo + hi) / 2, "n": len(sel),
                     "hr_acc_pct": 100 * np.mean(hr_vals) if hr_vals else np.nan,
                     "spo2_acc_pct": 100 * np.mean(sp_vals) if sp_vals else np.nan,
                     "overall_acc_pct": 100 * np.mean(both) if both else np.nan})
    df = pd.DataFrame(rows)
    out = os.path.join(DATASET_DIR, f"{prefix}_coverage_bins.csv")
    df.to_csv(out, index=False)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(df["bin_mid_pct"], df["overall_acc_pct"], marker="o")
    ax.set(xlabel="Monitor screen coverage (% of image area)",
           ylabel="Vital extraction accuracy (%)", ylim=(0, 100))
    ax.grid(alpha=0.3)
    plt.show()
    print(f"Saved {out}")
    return df

## Detector

In [ ]:
PROJECT, RUN_NAME = "yolo_monitor", "seg"
EPOCHS, BATCH, IMGSZ, CONF = 30, 8, 640, 0.25


def _weights_path():
    for p in (os.path.join("runs", "segment", PROJECT, RUN_NAME, "weights", "best.pt"),
              os.path.join(PROJECT, RUN_NAME, "weights", "best.pt")):
        if os.path.exists(p):
            return p
    return None


def load_detector(force_train=False):
    weights = _weights_path()
    if weights and not force_train:
        print(f"[YOLO] loading {weights}")
        return YOLO(weights)
    print("[YOLO] training yolov8n-seg on", DATA_YAML)
    model = YOLO("yolov8n-seg.pt")
    model.train(data=DATA_YAML, epochs=EPOCHS, batch=BATCH, imgsz=IMGSZ,
                project=PROJECT, name=RUN_NAME, exist_ok=True)
    return YOLO(_weights_path())


def detect(model, image_bgr, conf=CONF):
    res = model.predict(image_bgr, conf=conf, verbose=False)[0]
    polys = res.masks.xy if res.masks is not None else None
    dets = []
    for i, box in enumerate(res.boxes):
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cid = int(box.cls[0])
        poly = polys[i] if polys is not None and i < len(polys) else \
            np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]], dtype=float)
        dets.append({"cls": CLASS_NAMES[cid] if cid < len(CLASS_NAMES) else str(cid),
                     "bbox": [x1, y1, x2, y2], "poly": poly, "conf": float(box.conf[0])})
    return dets


def best(dets, cls):
    cands = [d for d in dets if d["cls"] == cls]
    return max(cands, key=lambda d: d["conf"]) if cands else None


def poly_crop(image_bgr, poly, pad=6, mask_outside=True):
    h, w = image_bgr.shape[:2]
    pts = np.asarray(poly, dtype=np.int32)
    x1, y1 = max(0, pts[:, 0].min() - pad), max(0, pts[:, 1].min() - pad)
    x2, y2 = min(w, pts[:, 0].max() + pad), min(h, pts[:, 1].max() + pad)
    if x2 <= x1 or y2 <= y1:
        return None
    region = image_bgr
    if mask_outside:
        m = np.zeros((h, w), np.uint8)
        cv2.fillPoly(m, [pts], 255)
        region = cv2.bitwise_and(image_bgr, image_bgr, mask=m)
    return region[y1:y2, x1:x2]


detector = load_detector(force_train=False)

## Molmo2-4B

In [ ]:
MOLMO_PROMPT = (
    "You are analyzing a medical patient monitor display image. Look carefully at all "
    "numbers visible on the screen and identify: 1. Heart Rate (HR) - typically labeled "
    "HR, 40-200 bpm. 2. SpO2 - oxygen saturation percentage, 70-100%. "
    'Respond ONLY with valid JSON: {"heart_rate": <int/null>, "spo2": <int/null>}'
)


def load_molmo(name=MOLMO_MODEL):
    processor = AutoProcessor.from_pretrained(name, trust_remote_code=True)
    model = AutoModelForImageTextToText.from_pretrained(
        name, trust_remote_code=True, dtype=torch.bfloat16, device_map="cuda")
    return model, processor


def _as_int(value, lo, hi):
    try:
        value = int(value)
    except (TypeError, ValueError):
        return None
    return value if lo <= value <= hi else None


def _parse_vitals(text):
    hr = spo2 = None
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m:
        try:
            data = json.loads(m.group(0))
            hr, spo2 = data.get("heart_rate"), data.get("spo2")
        except json.JSONDecodeError:
            pass
    if hr is None or spo2 is None:
        nums = [int(n) for n in re.findall(r"\d+", text)]
        hr = hr if hr is not None else next((n for n in nums if 40 <= n <= 200), None)
        spo2 = spo2 if spo2 is not None else next((n for n in nums if 70 <= n <= 100), None)
    return _as_int(hr, *HR_RANGE), _as_int(spo2, *SPO2_RANGE)


def read_vitals_molmo(monitor_bgr, model, processor):
    image = Image.fromarray(cv2.cvtColor(monitor_bgr, cv2.COLOR_BGR2RGB))
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image}, {"type": "text", "text": MOLMO_PROMPT}]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True)
    inputs = {k: (v.to(model.device, dtype=torch.bfloat16)
                  if torch.is_floating_point(v) else v.to(model.device))
              for k, v in inputs.items()}
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=128)
    text = processor.tokenizer.decode(
        generated[0, inputs["input_ids"].size(1):], skip_special_tokens=True)
    return _parse_vitals(text)


molmo_model, molmo_processor = load_molmo()

## Run the evaluation split

In [ ]:
PIPELINE_NAME = "Pipeline 3: YOLOv8 + Molmo2-4B"
IMAGES_DIR = os.path.join(DATASET_DIR, EVAL_SPLIT, "images")
LABELS_DIR = os.path.join(DATASET_DIR, EVAL_SPLIT, "labels")


def run_split(images_dir=IMAGES_DIR):
    readings = []
    for name in list_images(images_dir):
        img = cv2.imread(os.path.join(images_dir, name))
        if img is None:
            continue
        pm = best(detect(detector, img), "PM")
        hr = spo2 = None
        if pm is not None:
            crop = poly_crop(img, pm["poly"], pad=0, mask_outside=False)
            if crop is not None and crop.size:
                hr, spo2 = read_vitals_molmo(crop, molmo_model, molmo_processor)
        readings.append({"image_file": name, "bbox": pm["bbox"] if pm else None,
                         "hr": hr, "spo2": spo2})
    return readings


readings = run_split()
print(f"{len(readings)} images processed")

In [ ]:
example = next(r for r in readings if r["bbox"] is not None)
img = cv2.imread(os.path.join(IMAGES_DIR, example["image_file"]))
x1, y1, x2, y2 = example["bbox"]
cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)
plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title(f"HR {example['hr']}   SpO2 {example['spo2']}")
plt.axis("off")
plt.show()